## Install and load libraries:-

In [1]:
!pip install -q transformers faiss-gpu pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 13.1 MB/s eta 0:00:0000:0100:01


In [2]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import faiss
from transformers import CLIPProcessor, CLIPModel

## Load dataset:-

In [3]:
import os
import pandas as pd

# Automatically locate styles.csv and images directory
CSV_PATH = None
IMAGES_DIR = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "styles.csv" in files and CSV_PATH is None:
        CSV_PATH = os.path.join(root, "styles.csv")
    if "images" in dirs and IMAGES_DIR is None:
        IMAGES_DIR = os.path.join(root, "images")

if not CSV_PATH:
    raise FileNotFoundError("Could not find styles.csv in /kaggle/input/. Did you add the dataset to the notebook?")

print(f"Using CSV at: {CSV_PATH}")
print(f"Using Images from: {IMAGES_DIR}")

# Load metadata
df = pd.read_csv(CSV_PATH, on_bad_lines='skip')

# Attach full file path
df['image_path'] = df['id'].apply(lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg"))

# Filter out rows where the image file is missing on disk
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(f"Successfully loaded {len(df)} items.")

Using CSV at: /kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/styles.csv
Using Images from: /kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/images
Successfully loaded 44419 items.


## Initialize OpenAI CLIP Model:-

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

Using device: cuda


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

## Memory-Safe Full Extraction Function(Extract Embeddings)

In [5]:
# Cell 5: Safe Extraction Function
import gc

def extract_all_image_embeddings(df, batch_size=64):
    image_embeddings = []
    
    for i in tqdm(range(0, len(df), batch_size), desc="Extracting Embeddings"):
        batch_paths = df['image_path'].iloc[i:i+batch_size].tolist()
        
        images = []
        for path in batch_paths:
            try:
                img = Image.open(path).convert('RGB')
                images.append(img)
            except Exception:
                images.append(Image.new('RGB', (224, 224)))
        
        inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad():
            outputs = model.get_image_features(**inputs)
            
            # Check if outputs is a Tensor or a Hugging Face Object
            if hasattr(outputs, 'image_embeds'):
                features = outputs.image_embeds
            elif hasattr(outputs, 'pooler_output'):
                features = outputs.pooler_output
            else:
                features = outputs
                
            # Normalize embeddings (Cosine Similarity)
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            
        image_embeddings.append(features.cpu().numpy().astype('float32'))
        
        if (i // batch_size) % 10 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
    return np.vstack(image_embeddings)

image_vectors = extract_all_image_embeddings(df, batch_size=64)
print(f"Extraction complete! Embeddings shape: {image_vectors.shape}")

Extracting Embeddings: 100%|██████████| 695/695 [45:01<00:00,  3.89s/it]


Extraction complete! Embeddings shape: (44419, 512)


## Build FAISS Vector Index:-

In [6]:
dimension = image_vectors.shape[1]

# IndexFlatIP computes exact Cosine Similarity on normalized vectors
index = faiss.IndexFlatIP(dimension)
index.add(image_vectors.astype('float32'))

print(f"FAISS Index successfully created with {index.ntotal} items!")

FAISS Index successfully created with 44419 items!


## Define Search Functionality:-

In [7]:
def search_products(text_query, top_k=5):
    inputs = processor(text=[text_query], return_tensors="pt", padding=True).to(device)
    
    with torch.no_grad():
        outputs = model.get_text_features(**inputs)
        
        # Safely extract tensor
        if hasattr(outputs, 'text_embeds'):
            text_features = outputs.text_embeds
        elif hasattr(outputs, 'pooler_output'):
            text_features = outputs.pooler_output
        else:
            text_features = outputs
            
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        
    text_vector = text_features.cpu().numpy().astype('float32')
    scores, indices = index.search(text_vector, top_k)
    
    results = df.iloc[indices[0]].copy()
    results['similarity_score'] = scores[0]
    return results[['id', 'productDisplayName', 'articleType', 'baseColour', 'similarity_score', 'image_path']]

## Perform a Test Query:-

In [8]:
# Test search
results = search_products("black casual sneakers", top_k=5)
results

,id,productDisplayName,articleType,baseColour,similarity_score,image_path
7970,49381,Vans Men Navy Blue Versa Casual Shoes,Casual Shoes,Navy Blue,0.317582,/kaggle/input/datasets/paramaggarwal/fashion-p...
35449,49352,Vans Men Black Larkin Shoes,Casual Shoes,Black,0.315751,/kaggle/input/datasets/paramaggarwal/fashion-p...
19292,32744,Fila Men Dale Charcoal Black Shoes,Casual Shoes,Charcoal,0.314081,/kaggle/input/datasets/paramaggarwal/fashion-p...
42016,21735,Vans Men Pro Classics Black Casual Shoes,Casual Shoes,Black,0.313744,/kaggle/input/datasets/paramaggarwal/fashion-p...
25668,32193,ADIDAS Men Ohne Black Shoes,Casual Shoes,Black,0.313495,/kaggle/input/datasets/paramaggarwal/fashion-p...


## Calculate Evaluation Metrics (Precision@K & MRR)

In [9]:
def evaluate_pipeline(df, index, image_vectors, top_k=5, num_samples=200):
    np.random.seed(42)
    sample_indices = np.random.choice(len(df), size=min(num_samples, len(df)), replace=False)
    
    precisions = []
    reciprocal_ranks = []
    
    for idx in sample_indices:
        query_vec = image_vectors[idx:idx+1].astype('float32')
        target_category = df.iloc[idx]['articleType']
        
        scores, indices = index.search(query_vec, top_k + 1)
        retrieved_indices = indices[0][1:]
        
        matches = [df.iloc[r]['articleType'] == target_category for r in retrieved_indices]
        
        precisions.append(sum(matches) / top_k)
        
        rr = 0.0
        for rank, is_match in enumerate(matches, start=1):
            if is_match:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)
        
    print("=== PERFORMANCE EVALUATION METRICS ===")
    print(f"Precision@{top_k}: {np.mean(precisions) * 100:.2f}%")
    print(f"Mean Reciprocal Rank (MRR): {np.mean(reciprocal_ranks):.4f}")

evaluate_pipeline(df, index, image_vectors, top_k=5)

=== PERFORMANCE EVALUATION METRICS ===
Precision@5: 84.70%
Mean Reciprocal Rank (MRR): 0.9454


## Export FAISS Index & Embeddings for Deployment:-

In [10]:
# Save FAISS Index
faiss.write_index(index, "fashion_faiss.index")

# Save Embeddings Array
np.save("image_embeddings.npy", image_vectors)

# Save Filtered Dataset
df.to_csv("processed_styles.csv", index=False)

print("All deployment artifacts saved successfully!")

All deployment artifacts saved successfully!
